# Reseek SCOP40 all-vs-all: homologs found at 0.01, E-value and P-value vs SG p-value

The same experiment as `foldseek_scop40.ipynb`, for Reseek. Every ASTRAL40 domain (SCOPe 2.08,
15,177 structures) is searched against SCOP40 with **one** `src/reseek.py` call.
The wrapper takes a multi-chain query `.bca` and does the rest:
- it makes 50 decoys for every query (`-num_shuffles`);
- it searches the decoys and the real queries;
- it gives every hit an SG p-value computed from its AQ score.

SCOP labels provide the ground truth. A hit is a **homolog** if query and target share a
superfamily or family, and a **non-homolog** if they come from different folds. At the
cutoff 0.01, we count how many homologs and non-homologs Reseek's E-value and P-value and
the SG p-value each report.

| Step | What | Output |
|---|---|---|
| 1 | query database: every ASTRAL40 structure in one `.bca` | `results/scop40_run/reseek/query_db/queries.bca` |
| 2 | `reseek.py` on the query database vs `scop_target.bca` | `results/scop40_run/reseek/all_vs_all_sgpval.tsv[.gz]` |
| 3 | homologs and non-homologs found at 0.01 | table |

**Before you start:** run this notebook from `tutorial/` or the repository root. From
Zenodo, unpack these at the repository root:
- `results_scop40_run.tar.gz` holds the precomputed search results. With it, only step 3
  runs, in a few minutes.
- To search from scratch, you also need `data_scop40_run.tar.gz` (structures, labels and
  Reseek target database in `data/scop40_run/`) and the tools from `bash scripts/build.sh`.
  The paper's run took about 10 days.

Reseek decoys cannot be seeded, so a new search gives slightly different SG p-values.
With `USE_CACHED = True` (the default), a step is skipped when its output exists. The
table after the parameters shows what will run.

## Parameters

In [1]:
from pathlib import Path
import os
if not Path('scripts/build.sh').exists():   # opened from tutorial/: work from the repository root
    os.chdir('..')
ROOT = Path('.')                            # all paths below are relative to the repository root

RESEEK     = ROOT / 'bin/reseek'
DATA_DIR   = ROOT / 'data/scop40_run'       # inputs (Zenodo data_scop40_run.tar.gz)
RUN_DIR    = ROOT / 'results/scop40_run/reseek'   # results (Zenodo results_scop40_run.tar.gz)
PDBSTYLE   = DATA_DIR / 'pdbstyle-2.08'
TARGET_BCA = DATA_DIR / 'reseek/scop_target.bca'
FASTA      = DATA_DIR / 'astral.fa'         # ASTRAL40 ids with SCOP labels
LOOKUP     = ROOT / 'data/lookup/m50.txt'

NUM_SHUFFLE = 50      # decoys per query; must match LOOKUP (m50 -> 50)
EVALUE      = 100     # reseek -evalue
THREADS     = 32
CUTOFF      = 0.01
ROUND_SG    = True    # round SG p-values to 3 significant digits before the cutoff, as in the paper's counts
USE_CACHED  = True    # reuse results already in RUN_DIR (e.g. the Zenodo results); False = search again

In [2]:
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, 'src')
from foldseek import get_original_id, get_seq_label_map   # noqa: E402


def run(cmd, log=None):
    """Run a command, keep its output in `log`, fail loudly."""
    cmd = [str(c) for c in cmd]
    print('$', ' '.join(cmd), flush=True)
    with open(log, 'w') if log else open(os.devnull, 'w') as fh:
        subprocess.run(cmd, check=True, stdout=fh, stderr=subprocess.STDOUT)


def result_file():
    """The wrapper output (gzipped as shipped on Zenodo, or as written by step 2)."""
    for f in [RUN_DIR / 'all_vs_all_sgpval.tsv.gz', RUN_DIR / 'all_vs_all_sgpval.tsv']:
        if f.exists() and f.stat().st_size > 0:
            return f
    return RUN_DIR / 'all_vs_all_sgpval.tsv'


search_needed = not USE_CACHED or not result_file().exists()
pd.DataFrame([{'search results': 'cached' if not search_needed else '-',
               'will run': 'search (~10 days) + counting' if search_needed else 'counting only'}])

,search results,will run
0,cached,counting only


## 1. Query database

`reseek -convert` turns a directory of structures into one multi-chain `.bca`. It keeps one
chain per structure (model 1 of an NMR ensemble) and skips files without atom records. The
query set is the ASTRAL40 ids in `astral.fa`; 49 of them have no usable structure (48
missing from pdbstyle-2.08, 1 without atoms), which leaves 15,128 queries.

In [3]:
QUERY_BCA = RUN_DIR / 'query_db/queries.bca'

if search_needed and not QUERY_BCA.exists():
    sdir = QUERY_BCA.parent / 'structures'
    sdir.mkdir(parents=True, exist_ok=True)
    n = 0
    for qid in (l[1:].split()[0] for l in open(FASTA) if l.startswith('>')):
        f = PDBSTYLE / qid[2:4] / f'{qid}.ent'
        if f.exists():
            (sdir / f.name).unlink(missing_ok=True)
            (sdir / f.name).symlink_to(f.resolve())
            n += 1
    print(f'{n:,} ASTRAL40 structures found')
    run([RESEEK, '-convert', sdir, '-bca', QUERY_BCA], log=QUERY_BCA.parent / 'convert.log')
elif QUERY_BCA.exists():
    print(f'cached: {QUERY_BCA}')
else:
    print('query database not needed (search results cached)')

query database not needed (search results cached)


## 2. Search: one wrapper call

The settings are:
- **Decoys:** 50 per query, `-num_shuffles 50`.
- **Both searches:** `-verysensitive -evalue 100` against `scop_target.bca`.

The output has one row per hit: `aq query target evalue pvalue sgpval`. The decoy hits stay
in `results/scop40_run/reseek/reseek_output_*/`; delete them when done.

In [4]:
if search_needed:
    run([sys.executable, 'src/reseek.py', '--reseek_url', RESEEK, '--lookup_url', LOOKUP,
         '--num_shuffle', NUM_SHUFFLE, 'search', '--search', QUERY_BCA, '--db', TARGET_BCA,
         '--output', RUN_DIR / 'all_vs_all.tsv', '--evalue', EVALUE, '--verysensitive', '--threads', THREADS],
        log=RUN_DIR / 'wrapper.log')
else:
    print(f'cached: {result_file()}')

pd.read_csv(result_file(), sep='\t', nrows=5)

cached: results/scop40_run/reseek/all_vs_all_sgpval.tsv.gz


,aq,query,target,evalue,pvalue,sgpval
0,1.0000,d12asa__A,d12asa__A,0.000000e+00,0.000000e+00,8.716000e-15
1,0.9663,d12asa__A,d3p8ta__A,4.810000e-22,5.770000e-26,8.716000e-15
2,0.9418,d12asa__A,d3nema2_A,8.310000e-19,9.960000e-23,8.716000e-15
3,0.9410,d12asa__A,d1x54a2_A,1.010000e-18,1.210000e-22,8.716000e-15
4,0.9277,d12asa__A,d6hcwa2_A,1.680000e-17,2.010000e-21,8.716000e-15


## 3. Homologs and non-homologs found at 0.01

The counting rules are:
- **Counted:** query–target pairs; self-hits and pairs without SCOP labels are excluded.
- **Homolog:** same superfamily or family.
- **Non-homolog:** different fold.
- **Reported at the cutoff:** a pair is reported when its score is ≤ `CUTOFF`.
- **Rounding (`ROUND_SG = True`):** SG p-values are first rounded to 3 significant digits
  (`1.00e-02`), as in the wrapper's labelled output (`--fasta_url`) that the paper counted.
  Without rounding, 20 homologs and 5 non-homologs with SG p-values just above 0.01 are
  not counted. E-values and P-values are used as Reseek reports them.

With rounding these are the counts given in the paper.

In [5]:
labels = get_seq_label_map(FASTA)            # ASTRAL id -> (fold, superfamily, family)

df = pd.read_csv(result_file(), sep='\t', usecols=['query', 'target', 'evalue', 'pvalue', 'sgpval'])
df['sgpval'] = pd.to_numeric(df['sgpval'], errors='coerce')         # NaN = query without usable decoy scores
if ROUND_SG:                                                         # 3 significant digits, e.g. 0.01003 -> 0.0100
    df['sgpval'] = df['sgpval'].map({v: float(f'{v:.2e}') for v in pd.unique(df['sgpval'].dropna())})
df = df[~df['target'].str.contains('.', regex=False)]                # malformed ids (e.g. d1pvc.1_4)
norm = {i: get_original_id(i) for i in pd.unique(pd.concat([df['query'], df['target']]))}   # d1abc__A -> d1abc_
q, tg = df['query'].map(norm), df['target'].map(norm)
keep = (q.map(labels).notna() & tg.map(labels).notna() & (q != tg)).to_numpy()            # labelled, not self
df, q, tg = df[keep], q[keep], tg[keep]
ql, tl = np.array(q.map(labels).tolist()), np.array(tg.map(labels).tolist())
homolog = ql[:, 1] == tl[:, 1]                                       # same superfamily (or family)
nonhomolog = ql[:, 0] != tl[:, 0]                                    # different fold

table = pd.DataFrame([{'score': name, 'homologs found': int(((df[m] <= CUTOFF).to_numpy() & homolog).sum()),
                       'non-homologs found': int(((df[m] <= CUTOFF).to_numpy() & nonhomolog).sum())}
                      for m, name in [('evalue', 'E-value'), ('pvalue', 'P-value'), ('sgpval', 'SG p-value')]])
print(f'{len(df):,} labelled query-target pairs from {q.nunique():,} queries')
table

  0%|          | 0/15177 [00:00<?, ?it/s]

100%|██████████| 15177/15177 [00:00<00:00, 214286.99it/s]

32,163,746 labelled query-target pairs from 15,093 queries


,score,homologs found,non-homologs found
0,E-value,590128,2281875
1,P-value,818788,29346137
2,SG p-value,411889,10543


In [6]:
# change from each native score to the SG p-value at the same cutoff
sg = table.set_index('score').loc['SG p-value']
pd.DataFrame([{'SG p-value vs': name, **{
    col: f'{sg[col] - r[col]:+,} ({(sg[col] - r[col]) / r[col]:+.1%})' for col in ['homologs found', 'non-homologs found']}}
    for name, r in table.set_index('score').drop('SG p-value').iterrows()])

,SG p-value vs,homologs found,non-homologs found
0,E-value,"-178,239 (-30.2%)","-2,271,332 (-99.5%)"
1,P-value,"-406,899 (-49.7%)","-29,335,594 (-100.0%)"
